# // Pip Install all Required Libraries //

The cell below installs extra required python libraries

In [ ]:
!pip install vedo scikit-image h5py tifffile networkx

# // Imports //

The cell below imports all the required libraries for this tutorial

In [ ]:
import sys
import os
import numpy as np # Now safe to import

from pathlib import Path
from image_proc_funcs import IlastikClassifier
from image_proc_funcs import run_image_to_model

###########################################
### // IMPORT ALL REQUIRED LIBRARIES // ###
###########################################

# # --- CRITICAL FIX: PREVENT DEADLOCKS ---
# # Force all math libraries to use 1 thread to stop freezing
# os.environ['OPENBLAS_NUM_THREADS'] = '1'  <-- Comment this out
# os.environ['MKL_NUM_THREADS'] = '1'       <-- Comment this out
# os.environ['OMP_NUM_THREADS'] = '1'       <-- Comment this out
# os.environ['NUMEXPR_NUM_THREADS'] = '1'   <-- Comment this out

# Force Matplotlib to not use a window
os.environ['MPLBACKEND'] = 'Agg'  # <--- KEEP THIS! (Prevents GUI crashes in Docker)
# ---------------------------------------

# // Setting up Required Filepaths //

The cell below initialises all required file paths for the tutorial.

In [ ]:
# get dir where this file is. This should work locally or in Docker
this_dir = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
CA_root = Path(__file__).resolve().parent.parent.parent if "__file__" in globals() else Path.cwd().parent.parent
resources_path = CA_root / Path("resources")

#################### TODO make this automatic in Docker ####################################

src_path = os.path.join(CA_root, "src")
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

ilastik_path = Path("/usr/local/bin/ilastik")
model_path = CA_root / Path("tutorial/interactive/resources/image_process/segmentation_model.ilp")

input_batch_processing_path_test = CA_root / Path("tutorial/interactive/resources/image_process/batch_process_input_folder_TEST")
output_batch_processing_path_test = CA_root / Path("tutorial/interactive/resources/image_process/batch_process_output_folder_TEST")

#################################################
### // Initialise target image files paths // ###
#################################################

image_path_0 = CA_root / Path("tutorial/interactive/resources/image_process/batch_process_output_folder/image_0.h5") # Image 0
image_path_1 = CA_root / Path("tutorial/interactive/resources/image_process/batch_process_output_folder/image_1.h5") # Image 1
image_path_2 = CA_root / Path("tutorial/interactive/resources/image_process/batch_process_output_folder/image_2.h5") # Image 2
image_path_list = np.array([image_path_0, image_path_1, image_path_2])

#####################################
### // Initialise image target // ###
#####################################

### TODO: User to edit image_selection_index depending on which image (Image 0, Image 1, or Image 2) they would like to process.
image_selection_index = 0 ### Change value to 0, 1, or 2 depending on which image you would like to input into the pipeline
target_image_path = image_path_list[image_selection_index]

################################################
### // Initialise file paths for pipeline // ###
################################################

ilastik_path = Path("/opt/ilastik/run_ilastik.sh") ### TODO: Delete/uncomment after docker integration is working
model_path = CA_root / Path("tutorial/interactive/resources/image_process/segmentation_model.ilp")
input_batch_processing_path = CA_root / Path("tutorial/interactive/resources/image_process/batch_process_input_folder")
output_batch_processing_path = CA_root / Path("tutorial/interactive/resources/image_process/batch_process_output_folder")
run_ilastik_batch_processing = False ### TODO: Change to True if you want to use in-python Ilastik image segmentation

print('paths set')

# Image to Hemodynamics Model

This template walks through turning a microvasculature confocal microscopy image into a hemodynamics model, then exploring constriction (sympathetic activity) effects in PhLynx and WebOpenCOR.

## 1. Open ImageJ and visualize the image

1. Launch ImageJ (or Fiji).


In [ ]:
# Launch ImageJ/Fiji (requires X11 forwarding)
# On Linux OS, or Windows uncomment and run the line below
!/opt/fiji/fiji-linux-x64
# On Mac OS, uncomment and run the line below 
# !ln -sf /opt/fiji/java/linux64 /opt/fiji/java/linux-amd64 2>/dev/null; ln -sf /opt/fiji/java/linux64 /opt/fiji/java/linux-x64 2>/dev/null; export CONDA_PREFIX=/opt/conda CONDA_ROOT=/opt/conda CJDK_CACHE_DIR=/tmp XDG_CACHE_HOME=/tmp HOMEBREW_PREFIX=/tmp MAMBA_ROOT_PREFIX=/tmp JAVA_HOME=/opt/fiji/java/linux64; /opt/fiji/fiji-linux-x64

2. Open the source image: `File -> Open...` `circulatory_autogen/tutorial/interactive/resources/image_process/raw_image_files/Zstack1_Animal2_NG2_dsRed_CD31_647_GLUT_15042025/C2-Zstack1_Animal2_NG2_dsRed_CD31_647_GLUT_15042025_vessels.tif`
3. Verify the scale and orientation are correct (Image -> ShowInfo)
4. (Optional) Adjust contrast (Image -> Adjust -> Brightness/Contrast) or apply filters to clarify vessel boundaries.
When adjusting contrast, the changes are only superficial until apply is selected within Brightness/Contrast. It is generally inadvisable to permanently apply contrast changes and this should be used cosmetically).
In turns of filters, background subtract (Process -> Subtract Background) is a commonly used methodology to remove background noise (https://imagejdocu.list.lu/gui/process/subtract_background). Commonly used filters for vascular analysis include Guassian blur 3D (Process -> Filters -> Gaussian Blur 3D) and Frangi Vesselness (Process -> Filters -> Frangi Vesselness). These processes (amongst others) can improve vessel segmentation downstream, but classifiers should be trained on datasets using a chosen combination of image processing, and a chosen pipeline should be consistent amongst datasets that are directly compared.
5. (Optional) View image in 3D using Plugins -> BigDataViewer -> Open Current Image
6. (Optional) "Z-Project" to produce a projected 2D image from the 3D image series. This is typically how images are displayed within published papers in order to simplify the data (Image -> Stacks -> Z-Project -> Projection Type = Max intensity).
7. If you did any of the optional steps, close all open windows (without saving) and reopen the source image before proceeding. `File -> Open...` `circulatory_autogen/tutorial/interactive/resources/image_process/raw_image_files/Zstack1_Animal2_NG2_dsRed_CD31_647_GLUT_15042025/C2-Zstack1_Animal2_NG2_dsRed_CD31_647_GLUT_15042025_vessels.tif`

---- The instructions in this cell lets you see the vessel geometry that will be converted into a model but is not required for the next steps. ----

## 2. Convert image to `vessel_array.csv` and `parameters.csv`

Next we will run Python to process the image and generate two outputs:

- `vessel_array.csv` describing the vessel network connectivity and geometry
- `parameters.csv` defining model parameters

The conversion functions are not available yet, so the next cell contains placeholders.

In [ ]:
###################################################################
### // TEST CELL TO CHECK ILASTIK FUNCTIONALITY POST DOCKING // ###
###################################################################

# import sys
# from pathlib import Path
# from image_proc_funcs import IlastikClassifier

# circ_autogen_path = Path("/home/dsas627/PycharmProjects/circulatory_autogen")

# # ilastik_path = circ_autogen_path / Path("/path/to/ilastik") ### TODO: Change to appropriate path for docker testing
# ilastik_path = Path("/home/dsas627/Desktop/ilastik-1.4.1rc2-gpu-Linux/run_ilastik.sh") ### TODO: Delete/uncomment after docker integration is working
# model_path = circ_autogen_path / Path("tutorial/interactive/resources/image_process/segmentation_model.ilp")

# input_batch_processing_path_test = circ_autogen_path / Path("tutorial/interactive/resources/image_process/batch_process_input_folder_TEST")
# output_batch_processing_path_test = circ_autogen_path / Path("tutorial/interactive/resources/image_process/batch_process_output_folder_TEST")

# classifier = IlastikClassifier(ilastik_path, model_path)

# classifier.segment_images(input_dir=input_batch_processing_path_test,
#                           output_dir=output_batch_processing_path_test,
#                           input_ext="*.h5")

# print("Image Segmentation Completed.")

In [ ]:
##############################################################
### // Specify sub-volume (<= 0.15) of image to process // ###
##############################################################

### TODO: User to specify desired sub-volume. Ensure the value is <= 0.1 to avoid long processing times.
sub_volume = 0.1

##########################
### // Run pipeline // ###
##########################

run_image_to_model(target_image_path=target_image_path,
                   resources_path=resources_path,
                   ilastik_path=ilastik_path,
                   model_path=model_path,
                   input_batch_processing_path=input_batch_processing_path,
                   output_batch_processing_path=output_batch_processing_path,
                   sub_volume=sub_volume,
                   run_ilastik_batch_processing=run_ilastik_batch_processing)

print("Wrote:", str(CA_root / Path("tutorial/interactive/resources/user_output/image_to_model_vessel_array.csv")))
print("Wrote:", str(CA_root / Path("tutorial/interactive/resources/user_output/image_to_model_parameters.csv")))

## 3. Open the CSV files in PhLynx

1. Go to https://www.phlynx.com
2. Create a new project.
3. Import `vessel_array.csv` and `parameters.csv`.
4. Confirm the network and parameters are shown correctly.

## 4. Simulate in WebOpenCOR from PhLynx

1. From PhLynx, open the model in WebOpenCOR.
2. Run a baseline simulation.
3. Verify that pressures and flows are reasonable.

## 5. Add a sympathetic constriction module in PhLynx

1. Back in PhLynx, create a module that modifies vessel radius based on a sympathetic activity input.
2. Use a linear mapping, for example:

   `radius = baseline_radius * (1 - gain * sympathetic_activity)`

3. Connect the module to the target vessel(s).
4. Save the updated model.

## 6. Add a slider in WebOpenCOR to demonstrate sympathetic activity

1. Open the updated model in WebOpenCOR.
2. Create a slider bound to the sympathetic activity input.
3. Sweep the slider across its range and observe changes in pressure/flow.
4. Record any trends or screenshots for later analysis.